1. Spark UDFs
2. reading as dataframe API
3. reading as SPARK SQL
4. `help(spark.udf.register)`

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('Spark UDFs').getOrCreate()

24/04/02 22:20:49 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/04/02 22:20:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/02 22:20:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/02 22:20:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/04/02 22:20:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/04/02 22:20:51 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/04/02 22:20:51 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
help(spark.udf.register)

Help on method register in module pyspark.sql.udf:

register(name: str, f: Union[Callable[..., Any], ForwardRef('UserDefinedFunctionLike')], returnType: Union[ForwardRef('DataTypeOrString'), NoneType] = None) -> 'UserDefinedFunctionLike' method of pyspark.sql.udf.UDFRegistration instance
    Register a Python function (including lambda function) or a user-defined function
    as a SQL function.
    
    .. versionadded:: 1.3.1
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    name : str,
        name of the user-defined function in SQL statements.
    f : function, :meth:`pyspark.sql.functions.udf` or :meth:`pyspark.sql.functions.pandas_udf`
        a Python function, or a user-defined function. The user-defined function can
        be either row-at-a-time or vectorized. See :meth:`pyspark.sql.functions.udf` and
        :meth:`pyspark.sql.functions.pandas_udf`.
    returnType : :class:`pyspark.sql.types.DataType` or str, optional


In [4]:
df = spark.read.format('json').load('./retail_db_json/orders/', inferSchema=True)

In [5]:
df.dtypes

[('customer_city', 'string'),
 ('customer_email', 'string'),
 ('customer_fname', 'string'),
 ('customer_id', 'bigint'),
 ('customer_lname', 'string'),
 ('customer_password', 'string'),
 ('customer_state', 'string'),
 ('customer_street', 'string'),
 ('customer_zipcode', 'string'),
 ('department_id', 'bigint'),
 ('department_name', 'string'),
 ('order_customer_id', 'bigint'),
 ('order_date', 'string'),
 ('order_id', 'bigint'),
 ('order_status', 'string')]

In [6]:
df.show()

+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+---------------+----------------+-------------+---------------+-----------------+--------------------+--------+---------------+
|customer_city|customer_email|customer_fname|customer_id|customer_lname|customer_password|customer_state|customer_street|customer_zipcode|department_id|department_name|order_customer_id|          order_date|order_id|   order_status|
+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+---------------+----------------+-------------+---------------+-----------------+--------------------+--------+---------------+
|         NULL|          NULL|          NULL|       NULL|          NULL|             NULL|          NULL|           NULL|            NULL|         NULL|           NULL|            11599|2013-07-25 00:00:...|       1|         CLOSED|
|         NULL|          NULL|          NULL|       NULL|          N

In [7]:
from pyspark.sql.functions import isnotnull, col

In [8]:
df. \
    filter(isnotnull(col('customer_city'))==True). \
    show()

+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+--------------------+----------------+-------------+---------------+-----------------+----------+--------+------------+
|customer_city|customer_email|customer_fname|customer_id|customer_lname|customer_password|customer_state|     customer_street|customer_zipcode|department_id|department_name|order_customer_id|order_date|order_id|order_status|
+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+--------------------+----------------+-------------+---------------+-----------------+----------+--------+------------+
|  Brownsville|     XXXXXXXXX|       Richard|          1|     Hernandez|        XXXXXXXXX|            TX|  6303 Heather Plaza|           78521|         NULL|           NULL|             NULL|      NULL|    NULL|        NULL|
|    Littleton|     XXXXXXXXX|          Mary|          2|       Barrett|        XXXXXXXXX|          

In [34]:
sl = spark.udf.register('str_len', lambda x: len(x))

24/04/02 22:30:18 WARN SimpleFunctionRegistry: The function str_len replaced a previously registered function.


In [10]:
df.createOrReplaceTempView('customers')

In [37]:
spark.sql('''
    select str_len(customer_city) as length, 
    customer_city, 
    customer_email 
    from customers where customer_city is not null
''').show()

+------+-------------+--------------+
|length|customer_city|customer_email|
+------+-------------+--------------+
|    11|  Brownsville|     XXXXXXXXX|
|     9|    Littleton|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|    10|   San Marcos|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|     7|      Passaic|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|     8|     Lawrence|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|     8|     Stafford|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|    11|  San Antonio|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|    11|  Pico Rivera|     XXXXXXXXX|
|     7|      Fontana|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|     6|       Taylor|     XXXXXXXXX|
|     8|     Martinez|     XXXXXXXXX|
|     6|       Caguas|     XXXXXXXXX|
|    13|West New York|     XXXXXXXXX|
+------+-------------+--------------+
only showing top 20 rows



In [24]:
df.show()

+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+---------------+----------------+-------------+---------------+-----------------+--------------------+--------+---------------+
|customer_city|customer_email|customer_fname|customer_id|customer_lname|customer_password|customer_state|customer_street|customer_zipcode|department_id|department_name|order_customer_id|          order_date|order_id|   order_status|
+-------------+--------------+--------------+-----------+--------------+-----------------+--------------+---------------+----------------+-------------+---------------+-----------------+--------------------+--------+---------------+
|         NULL|          NULL|          NULL|       NULL|          NULL|             NULL|          NULL|           NULL|            NULL|         NULL|           NULL|            11599|2013-07-25 00:00:...|       1|         CLOSED|
|         NULL|          NULL|          NULL|       NULL|          N

In [45]:
df.filter(isnotnull(col('customer_city'))==True) \
    .selectExpr('str_len(customer_city) as city_length') \
    .show()

+-----------+
|city_length|
+-----------+
|         11|
|          9|
|          6|
|         10|
|          6|
|          7|
|          6|
|          8|
|          6|
|          8|
|          6|
|         11|
|          6|
|         11|
|          7|
|          6|
|          6|
|          8|
|          6|
|         13|
+-----------+
only showing top 20 rows



In [48]:
df.filter(isnotnull(col('customer_city'))==True) \
    .select(sl('customer_city')) \
    .show()

+----------------------+
|str_len(customer_city)|
+----------------------+
|                    11|
|                     9|
|                     6|
|                    10|
|                     6|
|                     7|
|                     6|
|                     8|
|                     6|
|                     8|
|                     6|
|                    11|
|                     6|
|                    11|
|                     7|
|                     6|
|                     6|
|                     8|
|                     6|
|                    13|
+----------------------+
only showing top 20 rows

